In [15]:
import json
import logging
import warnings
from pathlib import Path
from typing import List
from dataclasses import dataclass, asdict, field
from enum import Enum
import xml.etree.ElementTree as ET

from cltk.nlp import NLP


In [ ]:
# Configuration
logging.basicConfig(level=logging.INFO, format="%(message)s")
logger = logging.getLogger(__name__)

# Data Structures
class ChunkType(str, Enum):
    SENTENCE = "sentence"
    THREE_SENTENCE = "3-sentence"
    TOKEN_WINDOW = "150-token-window"


@dataclass
class Chunk:
    chunk_id: str
    text: str
    chunk_type: ChunkType
    source_file: str
    position: int
    token_count: int
    sentence_indices: List[int] = field(default_factory=list)

    def to_dict(self):
        data = asdict(self)
        data["chunk_type"] = self.chunk_type.value
        return data

# BDC CHUNKER (Letters)

class BDCChunker:
    def __init__(self, nlp: NLP):
        self.nlp = nlp

    def chunk_letter(self, letter_path: Path) -> List[Chunk]:

        text = letter_path.read_text(encoding="utf-8")
        letter_id = letter_path.stem

        doc = self.nlp.analyze(text)

        sentence_texts = []
        sentence_token_counts = []

        for sentence in doc.sentences:
            words = [
                word.string
                for word in sentence.words
                if word.string and word.string.strip()
            ]

            if not words:
                continue

            sentence_texts.append(" ".join(words))
            sentence_token_counts.append(len(words))

        if not sentence_texts:
            logger.warning(f"⚠ No sentences found in {letter_id}")
            return []

        chunks = []

        # 1. Single sentence chunks
        for i, (sent_text, token_count) in enumerate(
            zip(sentence_texts, sentence_token_counts)
        ):
            chunks.append(
                Chunk(
                    chunk_id=f"{letter_id}_sent_{i}",
                    text=sent_text,
                    chunk_type=ChunkType.SENTENCE,
                    source_file=letter_path.name,
                    position=i,
                    token_count=token_count,
                    sentence_indices=[i],
                )
            )

        # 2. Three-sentence overlapping windows (1-sentence step)
        for i in range(len(sentence_texts) - 2):
            combined_text = " ".join(sentence_texts[i:i+3])
            combined_tokens = sum(sentence_token_counts[i:i+3])

            chunks.append(
                Chunk(
                    chunk_id=f"{letter_id}_sent_{i}_{i+2}",
                    text=combined_text,
                    chunk_type=ChunkType.THREE_SENTENCE,
                    source_file=letter_path.name,
                    position=i,
                    token_count=combined_tokens,
                    sentence_indices=[i, i+1, i+2],
                )
            )

        return chunks

    def chunk_all_letters(self, letters_dir: Path, output_path: Path):

        letter_files = sorted(Path(letters_dir).glob("*.txt"))
        logger.info(f"Found {len(letter_files)} letters to chunk\n")

        all_chunks = []
        total_tokens = 0

        for letter_file in letter_files:
            logger.info(f"Processing {letter_file.name}...")
            chunks = self.chunk_letter(letter_file)

            total_tokens += sum(c.token_count for c in chunks)
            all_chunks.extend(chunks)

            logger.info(f"✓ {len(chunks)} chunks")

        total_chunks = len(all_chunks)
        avg_tokens = total_tokens / total_chunks if total_chunks else 0

        output_data = {
            "metadata": {
                "corpus": "BDC",
                "chunking_strategy": "sentence + 3-sentence overlapping windows",
                "total_letters": len(letter_files),
                "total_chunks": total_chunks,
                "avg_tokens_per_chunk": avg_tokens,
            },
            "chunks": [c.to_dict() for c in all_chunks],
        }

        output_path.write_text(
            json.dumps(output_data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        logger.info("\nBDC CHUNKING COMPLETE")
        logger.info(f"Total chunks: {total_chunks}")
        logger.info(f"Average tokens per chunk: {avg_tokens:.2f}\n")

        return all_chunks


# PSC CHUNKER (Patristic Sources)

class PSCChunker:
    def __init__(self, window_size=150, overlap=50):
        self.window_size = window_size
        self.overlap = overlap

    def extract_text_from_tei(self, xml_path: Path) -> str:
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            ns = {"tei": "http://www.tei-c.org/ns/1.0"}

            body = root.find(".//tei:body", ns)
            text = "".join(body.itertext()) if body is not None else "".join(root.itertext())

            return " ".join(text.split())

        except Exception as e:
            logger.warning(f"⚠ XML parse error in {xml_path.name}: {e}")
            return xml_path.read_text(encoding="utf-8")

    def chunk_source(self, source_path: Path) -> List[Chunk]:
        text = self.extract_text_from_tei(source_path)
        if not text.strip():
            return []
        
        # Whitespace tokenization
        tokens = text.split()
        
        if len(tokens) < self.window_size:
            return [
                Chunk(
                    chunk_id=f"{source_path.stem}_window_0",
                    text=" ".join(tokens),
                    chunk_type=ChunkType.TOKEN_WINDOW,
                    source_file=source_path.name,
                    position=0,
                    token_count=len(tokens),
                )
            ]

        chunks = []
        step = self.window_size - self.overlap

        for i, position in enumerate(range(0, len(tokens), step)):
            window_tokens = tokens[position : position + self.window_size]
            if not window_tokens:
                break

            chunks.append(
                Chunk(
                    chunk_id=f"{source_path.stem}_window_{i}",
                    text=" ".join(window_tokens),
                    chunk_type=ChunkType.TOKEN_WINDOW,
                    source_file=source_path.name,
                    position=position,
                    token_count=len(window_tokens),
                )
            )

            if position + self.window_size >= len(tokens):
                break

        return chunks

    def chunk_all_sources(self, sources_dir: Path, output_path: Path):
        source_files = sorted(Path(sources_dir).glob("*.xml"))
        logger.info(f"Found {len(source_files)} sources to chunk\n")

        all_chunks = []
        total_tokens = 0

        for source_file in source_files:
            logger.info(f"Processing {source_file.name}...")
            chunks = self.chunk_source(source_file)

            total_tokens += sum(c.token_count for c in chunks)
            all_chunks.extend(chunks)

            logger.info(f"✓ {len(chunks)} windows")

        total_chunks = len(all_chunks)
        avg_tokens = total_tokens / total_chunks if total_chunks else 0

        output_data = {
            "metadata": {
                "corpus": "PSC",
                "chunking_strategy": f"{self.window_size}-token sliding window with {self.overlap}-token overlap",
                "window_size": self.window_size,
                "overlap": self.overlap,
                "total_sources": len(source_files),
                "total_chunks": total_chunks,
                "avg_tokens_per_chunk": avg_tokens,
                "lemma_based": False,
            },
            "chunks": [c.to_dict() for c in all_chunks],
        }

        output_path.write_text(
            json.dumps(output_data, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

        logger.info("\nPSC CHUNKING COMPLETE")
        logger.info(f"Total chunks: {total_chunks}")
        logger.info(f"Average tokens per chunk: {avg_tokens:.2f}\n")

        return all_chunks


# MAIN

def main():

    letters_dir = Path("../annotations/evaluation-dataset/letters")
    sources_dir = Path("../annotations/evaluation-dataset/sources")
    output_dir = Path("../annotations/evaluation-dataset/chunks")
    output_dir.mkdir(exist_ok=True)

    logger.info("Initializing CLTK Latin NLP pipeline for BDC...\n")
    nlp = NLP(language="lat")

    bdc_chunker = BDCChunker(nlp)
    psc_chunker = PSCChunker(window_size=150, overlap=50)

    bdc_chunks = bdc_chunker.chunk_all_letters(
        letters_dir, output_dir / "bdc_chunks.json"
    )

    psc_chunks = psc_chunker.chunk_all_sources(
        sources_dir, output_dir / "psc_chunks.json"
    )

    logger.info("CHUNKING COMPLETE")
    logger.info(f"BDC chunks: {len(bdc_chunks):,}")
    logger.info(f"PSC chunks: {len(psc_chunks):,}")
    logger.info(f"Total chunks: {len(bdc_chunks) + len(psc_chunks):,}")


if __name__ == "__main__":
    main()

Initializing CLTK Latin NLP pipeline for BDC...

Found 67 letters to chunk

Processing 10015.txt...


‎𐤀 CLTK version '1.5.0'. When using the CLTK in research, please cite: https://aclanthology.org/2021.acl-demo.3/

Pipeline for language 'Latin' (ISO: 'lat'): `LatinNormalizeProcess`, `LatinStanzaProcess`, `LatinEmbeddingsProcess`, `StopsProcess`, `LatinLexiconProcess`.

⸖ ``LatinStanzaProcess`` using Stanza model from the Stanford NLP Group: https://stanfordnlp.github.io/stanza/ . Please cite: https://arxiv.org/abs/2003.07082
⸖ ``LatinEmbeddingsProcess`` using word2vec model by University of Oslo from http://vectors.nlpl.eu/ . Please cite: https://aclanthology.org/W17-0237/
⸖ ``LatinLexiconProcess`` using Lewis's *An Elementary Latin Dictionary* (1890).

⸎ To suppress these messages, instantiate ``NLP()`` with ``suppress_banner=True``.
Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'C

✓ 520 chunks
Processing 10041.txt...


Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error

✓ 720 chunks
Processing 10053.txt...


...

✓ 18 chunks
Processing 9183.txt...


Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error

✓ 92 chunks
Processing 9780.txt...


Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error in the dependency parser, please raise an issue at <https://github.com/cltk/cltk/issues> and include a short text to reproduce the error.

Unrecognized UD feature 'Compound' with value 'Yes'.
If you believe this is not an error

✓ 110 chunks

BDC CHUNKING COMPLETE
Total chunks: 18950
Average tokens per chunk: 39.06

Found 49 sources to chunk

Processing 001_Tertullianus_Apologeticus-adversus-gentes.xml...
✓ 236 windows
Processing 001_Tertullianus_De-idololatria.xml...
✓ 74 windows
Processing 001_Tertullianus_De-poenitentia.xml...
✓ 53 windows
Processing 002_Tertullianus_Adversus-Marcionem.xml...
✓ 901 windows
Processing 002_Tertullianus_De-praescriptionibus-adversus-haereticos.xml...
✓ 131 windows
Processing 002_Tertullianus_De-resurrectione-carnis.xml...
✓ 239 windows
Processing 003_Cyprianus-Carthaginensis_Epistola.xml...
✓ 6 windows
Processing 004_Cyprianus-Carthaginensis_Ad-Demetrianum.xml...
✓ 49 windows
Processing 004_Cyprianus-Carthaginensis_De-unitate-ecclesiae.xml...
✓ 56 windows
Processing 006_Lactantius_Divinarum-institutionum-liber-IV.xml...
✓ 182 windows
Processing 010_Hilarius-Pictaviensis_De-Trinitate.xml...
✓ 1302 windows
Processing 010_Hilarius-Pictaviensis_Liber-de-synodis-seu-fide-orientaliu

In [6]:
def count_tokens_in_chunks(chunks_json_path: Path):
    """Count total tokens in a chunks JSON file"""
    
    data = json.loads(chunks_json_path.read_text(encoding="utf-8"))
    
    chunks = data["chunks"]
    total_tokens = sum(chunk["token_count"] for chunk in chunks)
    total_chunks = len(chunks)
    avg_tokens = total_tokens / total_chunks if total_chunks else 0
   
    print(f"Total chunks: {total_chunks:,}")
    print(f"Total tokens: {total_tokens:,}")
    print(f"Average tokens per chunk: {avg_tokens:.2f}")
    print(f"Min tokens: {min(c['token_count'] for c in chunks)}")
    print(f"Max tokens: {max(c['token_count'] for c in chunks)}")
    
    # Count by chunk type
    from collections import Counter
    chunk_types = Counter(c["chunk_type"] for c in chunks)
    
    print(f"\nBy chunk type:")
    for chunk_type, count in chunk_types.items():
        type_tokens = sum(c["token_count"] for c in chunks if c["chunk_type"] == chunk_type)
        print(f"  {chunk_type}: {count:,} chunks, {type_tokens:,} tokens")
    
    return {
        "total_chunks": total_chunks,
        "total_tokens": total_tokens,
        "avg_tokens": avg_tokens
    }

if __name__ == "__main__":
    
    psc_path = Path("../annotations/evaluation-dataset/chunks/psc_chunks.json")
    bdc_path = Path("../annotations/evaluation-dataset/chunks/bdc_chunks.json")
    
    print("PSC CHUNKS:")
    psc_stats = count_tokens_in_chunks(psc_path)
    print("\n")
    
    print("BDC CHUNKS:")
    bdc_stats = count_tokens_in_chunks(bdc_path)
    

    print(f"Total tokens: {psc_stats['total_tokens'] + bdc_stats['total_tokens']:,}")
    print(f"Total chunks: {psc_stats['total_chunks'] + bdc_stats['total_chunks']:,}")

PSC CHUNKS:
Total chunks: 46,408
Total tokens: 6,959,056
Average tokens per chunk: 149.95
Min tokens: 58
Max tokens: 150

By chunk type:
  150-token-window: 46,408 chunks, 6,959,056 tokens


BDC CHUNKS:
Total chunks: 19,466
Total tokens: 757,043
Average tokens per chunk: 38.89
Min tokens: 1
Max tokens: 268

By chunk type:
  sentence: 9,800 chunks, 190,635 tokens
  3-sentence: 9,666 chunks, 566,408 tokens
Total tokens: 7,716,099
Total chunks: 65,874
